In [67]:
import pandas as pd
from collections import Counter
import numpy as np
from scipy.stats import entropy

In [68]:
def jsd_divergence_between_sets(set1, set2, base=2):

    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq

    return jsd_results

In [ ]:
group_var_dict = {"V201549x" : "Race", 'V202022' : "Discuss Politics", 'V201200' : "Ideology", 'V201231x' : "Party", 
                  'V201452' : "Church", 'V201600' : "Gender", 'V202406' : "Political Interest"}

groups = {
    'V201549x': {1: 'white', 2: 'black', 3: 'asian', 4: 'native American', 5: 'hispanic'},
    'V202022': {
            1: 'I like to discuss politics with my family and friends.',
            2: 'I never discuss politics with my family or friends.'
            },
    'V201200': {
            1: "extremely liberal",
            2: "liberal",
            3: "slightly liberal",
            4: "moderate",
            5: "slightly conservative",
            6: "conservative",
            7: "extremely conservative"
        },
    'V201231x': {
            1: "a strong democrat",
            2: "a weak Democrat",
            3: "an independent who leans Democratic",
            4: "an independent",
            5: "an independent who leans Republican",
            6: "a weak Republican",
            7: "a strong Republican"
        },
    'V201452': {1: "attend church", 2: "do not attend church"},
    'V201600': {1: "man", 2: "woman"},
    'V202406': {1: "very", 2: "somewhat", 3: "not very", 4: "not at all"}
}

In [ ]:
qids = ["V201324","V201416", "V202234", "V202257", "V202287", "V202332", "V202337", "V202348", "V202371", "V202378"]
name_dict = {"V201324" : "Current Economy", "V201416" : "Gay Marriage", "V202234" : "Refugee Allowing", 
            "V202257" : "Income Inequality", "V202287" : "Gender Role", "V202332": "Climate Change",
            "V202337": "Gun Regulation", "V202348": "Drug Addiction", "V202371": "Race Diversity", "V202378": "Health Insurance"}

groups_keys = groups.keys()

def get_counts_group(path, qids, key): #function to get a dict with counts for each question

    value_list = {}

    for number, group in groups[key].items():

        value_list[group] = []

        for id in qids:
            df = pd.read_csv(path + id + ".csv")
            df_group = df[df[key] == number]

            df_group.loc[:, 'Response'] = pd.to_numeric(df_group['Response'], errors='coerce')     
            value_counts = Counter(df_group['Response'].dropna())

            if id in ["V201324", "V202332"]:

                answers = {i: value_counts[i] for i in range(1, 6)}
            else:
                answers = {i: value_counts[i] for i in range(1, 4)}

            value_list[group].append({name_dict[id] : answers})

    return value_list

In [85]:
def get_human_counts_group(path, qids, key): #getting results from anes 2020
     
    value_list = {}

    for number, group in groups[key].items():

        value_list[group] = []

        df = pd.read_csv(path)

        df_group = df[df[key] == number]

        for id in qids:

            human_counts = Counter(df_group[id].dropna())

            if id in ["V201324", "V202332"]:

                answers = {i: human_counts[i] for i in range(1, 6)}
            else:
                answers = {i: human_counts[i] for i in range(1, 4)}

            value_list[group].append({name_dict[id] : answers})

    return value_list

#human_results = get_human_counts_group("data/2020 ANES_test.csv", qids)

In [86]:
human_results_groups = {}
replicate_results_groups = {}
reformulated_results_groups = {}

for key in groups_keys:
    human_results_groups[group_var_dict[key]] = get_human_counts_group("data/2020 ANES_test.csv", qids, key)
    replicate_results_groups[group_var_dict[key]] = get_counts_group("Publication Results/GPT4.1/main_mq_results/full_results_2020_", qids, key)
    reformulated_results_groups[group_var_dict[key]] = get_counts_group("Publication Results/GPT4.1/main_mq_reform_results/full_results_2020_", qids, key)

results = [human_results_groups, replicate_results_groups, reformulated_results_groups]    

C:\Users\msmir\AppData\Local\Temp\ipykernel_27328\3705012249.py:9: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
C:\Users\msmir\AppData\Local\Temp\ipykernel_27328\3705012249.py:9: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
C:\Users\msmir\AppData\Local\Temp\ipykernel_27328\3705012249.py:9: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
C:\Users\msmir\AppData\Local\Temp\ipykernel_27328\3705012249.py:9: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
C:\Users\msmir\AppData\Local\Temp\ipykernel_27328\370501

In [87]:
black_results = [result["Race"]["black"] for result in results]

diff_black = {key: (jsd_divergence_between_sets(black_results[0], black_results[2])[key] - jsd_divergence_between_sets(black_results[0], black_results[1])[key]) 
                    for key in name_dict.values()}

diff_black

{'Current Economy': np.float64(-0.03715370317105929),
 'Gay Marriage': np.float64(-0.003106232722617938),
 'Refugee Allowing': np.float64(-0.05330911249107159),
 'Income Inequality': np.float64(0.07987176752135977),
 'Gender Role': np.float64(-0.06947472748522343),
 'Climate Change': np.float64(0.008019277658719667),
 'Gun Regulation': np.float64(0.014959658790019259),
 'Drug Addiction': np.float64(-0.011377981119586295),
 'Race Diversity': np.float64(-0.08276442406167897),
 'Health Insurance': np.float64(0.015132710176217961)}

In [88]:
white_results = [result["Race"]["white"] for result in results]

diff_white = {key: (jsd_divergence_between_sets(white_results[0], white_results[2])[key] - jsd_divergence_between_sets(white_results[0], white_results[1])[key]) 
                    for key in name_dict.values()}

diff_white

{'Current Economy': np.float64(-0.059597163347859694),
 'Gay Marriage': np.float64(-0.009216386774197838),
 'Refugee Allowing': np.float64(-0.06020685555947191),
 'Income Inequality': np.float64(0.04324188362343401),
 'Gender Role': np.float64(-0.04810637359805787),
 'Climate Change': np.float64(-0.04227419202750589),
 'Gun Regulation': np.float64(0.012186274420453013),
 'Drug Addiction': np.float64(-0.023771832092110368),
 'Race Diversity': np.float64(-0.08220371484452146),
 'Health Insurance': np.float64(0.02320991174132346)}

In [77]:
asian_results = [result["Race"]["asian"] for result in results]

diff_asian = {key: (jsd_divergence_between_sets(asian_results[0], asian_results[2])[key] - jsd_divergence_between_sets(asian_results[0], asian_results[1])[key]) 
                    for key in name_dict.values()}

diff_asian

{'Current Economy': np.float64(0.0800733958411318),
 'Gay Marriage': np.float64(-0.012457517556742154),
 'Refugee Allowing': np.float64(-0.008652216050099838),
 'Income Inequality': np.float64(0.03560897648398112),
 'Gender Role': np.float64(0.059510406701320136),
 'Climate Change': np.float64(-0.0013500661894760985),
 'Gun Regulation': np.float64(0.011062481748877924),
 'Drug Addiction': np.float64(0.019893410469451426),
 'Race Diversity': np.float64(0.061393135162630626),
 'Health Insurance': np.float64(0.01101224719202053)}

In [ ]:
jsd_results_replicate = jsd_divergence_between_sets(black_results[0], black_results[1], base=2)
jsd_results_reformulated = jsd_divergence_between_sets(black_results[0], black_results[2], base=2)

diff_reformulated_vs_replicate = {key: (jsd_results_reformulated[key] - jsd_results_replicate[key]) for key in jsd_results_replicate.keys()}